# 01 · Ingestion des sources

In [1]:
%pip install -q geopandas xlrd openpyxl scikit-learn requests pyarrow

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.13/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
ROOT = "/lakehouse/default"

In [3]:
# Parameters
ROOT = "data/lakehouse"


In [4]:
import json, zipfile, io, gzip
from pathlib import Path
import requests
from urllib3.util import Retry
from shapely.geometry import shape
import pandas as pd
import geopandas as gpd

RAW = Path(ROOT) / "Files/hackaton/raw"
RAW.mkdir(parents=True, exist_ok=True)
DG = "https://www.data.gouv.fr/api/1/datasets/r/"
http = requests.Session()
http.mount("https://", requests.adapters.HTTPAdapter(max_retries=Retry(total=5, backoff_factor=2)))

def get(url, name):
    path = RAW / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        r = http.get(url, timeout=600)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path

def save(df, name):
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name, len(df))

In [5]:
villes = pd.DataFrame([
    ["Bordeaux", "243300316", "33", DG + "10b87ffe-e6bb-494d-93df-bb6019e223d9", DG + "4a69aee4-05ea-45e9-81b7-f9dbdfc7a344", DG + "51af208b-121d-48ae-b8ee-d45864e450ae"],
    ["Grenoble", "200040715", "38", DG + "b6ec7ba4-09bc-46df-b9a1-79a2c2668cf2", DG + "4553afe7-fdc4-4f0f-98e5-9d11b5039c57", None],
    ["Lille", "200093201", "59", DG + "c9e5dd3f-8eed-4ad7-aec2-915240599cf6", DG + "ea739e26-e3b8-4e93-84d3-31b12c12a10d", DG + "36459a88-73c3-47e6-acef-d1a00378c140"],
    ["Rennes", "243500139", "35", DG + "ef4779c4-d066-40d8-b8d9-fb8c7eea8820", DG + "9d526d49-1db5-48f8-baa0-a5b83991a222", DG + "ed5d8c45-4768-4e78-97f8-e0cd6c20e0a9"],
    ["Toulouse", "243100518", "31", DG + "64318177-f5b5-4144-a6be-2d0f22d26c77", DG + "a73a8d89-a43e-4d9b-883d-854b5864a4bf", "https://data.toulouse-metropole.fr/api/explore/v2.1/catalog/datasets/velotoulouse-localisation-et-caracteristique-des-stations/exports/csv?delimiter=%2C"],
], columns=["ville", "epci", "dep", "gtfs", "gbfs_citiz", "gbfs_velo"])
save(villes, "bronze_villes")

bronze_villes 5


In [6]:
communes = pd.concat([
    pd.DataFrame(requests.get(f"https://geo.api.gouv.fr/epcis/{v.epci}/communes?fields=code,nom", timeout=60).json()).assign(ville=v.ville)
    for v in villes.itertuples()
]).rename(columns={"code": "com", "nom": "nom_commune"})
save(communes, "bronze_communes")

bronze_communes 252


In [7]:
cols = ["POP", "POP0014", "POP1529", "POP6074", "POP75P"]

p21 = pd.read_csv(zipfile.ZipFile(get("https://www.insee.fr/fr/statistiques/fichier/8268806/base-ic-evol-struct-pop-2021_csv.zip", "pop2021.zip")).open("base-ic-evol-struct-pop-2021.CSV"), sep=";", dtype={"IRIS": str, "COM": str})
p21 = p21[p21.COM.isin(communes.com)][["IRIS", "COM"] + [f"P21_{c}" for c in cols]]
save(p21, "bronze_pop_2021")

p15 = pd.read_excel(zipfile.ZipFile(get("https://www.insee.fr/fr/statistiques/fichier/3627376/base-ic-evol-struct-pop-2015.zip", "pop2015.zip")).open("base-ic-evol-struct-pop-2015.xls"), sheet_name="IRIS", header=5, dtype={"IRIS": str, "COM": str})
p15 = p15[p15.COM.isin(communes.com)][["IRIS", "COM"] + [f"P15_{c}" for c in cols]]
save(p15, "bronze_pop_2015")

/tmp/ipykernel_1040138/2795248908.py:3: DtypeWarning: Columns (3: LAB_IRIS) have mixed types. Specify dtype option on import or set low_memory=False.
  p21 = pd.read_csv(zipfile.ZipFile(get("https://www.insee.fr/fr/statistiques/fichier/8268806/base-ic-evol-struct-pop-2021_csv.zip", "pop2021.zip")).open("base-ic-evol-struct-pop-2021.CSV"), sep=";", dtype={"IRIS": str, "COM": str})


bronze_pop_2021 1421


bronze_pop_2015 1420


In [8]:
om = pd.read_excel(get("https://www.insee.fr/fr/statistiques/fichier/6652134/projections_scenario_central.xlsx", "omphale_central.xlsx"), sheet_name="Population_DEP", header=5, dtype={"DEP": str})
om = om[om.DEP.isin(villes.dep)][["DEP", "LIBDEP", "POP_2021", "POP_2030"]]
save(om, "bronze_omphale")

bronze_omphale 5


In [9]:
WFS = "https://data.geopf.fr/wfs/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=STATISTICALUNITS.IRIS:contours_iris&outputFormat=application/json&CQL_FILTER=code_insee IN ({})"

iris = []
for v, g in communes.groupby("ville"):
    path = get(WFS.format(",".join(f"'{c}'" for c in g.com)), f"iris_{v}.geojson")
    iris.append(pd.json_normalize(json.loads(path.read_text())["features"]).assign(ville=v))
iris = pd.concat(iris)
iris["geometry_json"] = iris.apply(lambda r: json.dumps({"type": r["geometry.type"], "coordinates": r["geometry.coordinates"]}), axis=1)
iris = iris.rename(columns=lambda c: c.replace("properties.", ""))[["ville", "code_iris", "nom_iris", "code_insee", "nom_commune", "type_iris", "geometry_json"]]
save(iris, "bronze_iris_geo")

bronze_iris_geo 1422


In [10]:
stops = []
for v in villes.itertuples():
    s = pd.read_csv(zipfile.ZipFile(get(v.gtfs, f"gtfs_{v.ville}.zip")).open("stops.txt"), dtype=str)
    s = s[s.get("location_type", pd.Series("0", index=s.index)).fillna("0").isin(["0", ""])]
    stops.append(s[["stop_id", "stop_name", "stop_lat", "stop_lon"]].astype({"stop_lat": float, "stop_lon": float}).assign(ville=v.ville))
save(pd.concat(stops), "bronze_stops")

bronze_stops 15946


In [11]:
def gbfs_stations(url, name):
    if "exports/csv" in url:
        return pd.read_csv(get(url, name + ".csv"))[["station_id", "name", "lat", "lon"]].astype({"station_id": str})
    root = json.loads(get(url, name + ".json").read_text())["data"]
    feeds = root["feeds"] if "feeds" in root else next(iter(root.values()))["feeds"]
    info = next(f["url"] for f in feeds if f["name"] == "station_information")
    st = pd.DataFrame(requests.get(info, timeout=60).json()["data"]["stations"])
    st["name"] = st["name"].apply(lambda n: n[0]["text"] if isinstance(n, list) else n)
    return st[["station_id", "name", "lat", "lon"]].astype({"station_id": str})

citiz = pd.concat([gbfs_stations(v.gbfs_citiz, f"gbfs_citiz_{v.ville}").assign(ville=v.ville) for v in villes.itertuples()])
save(citiz, "bronze_citiz")
velo = pd.concat([gbfs_stations(v.gbfs_velo, f"velo_{v.ville}").assign(ville=v.ville) for v in villes.itertuples() if pd.notna(v.gbfs_velo)])
save(velo, "bronze_velo")

bronze_citiz 800


bronze_velo 1023


In [12]:
SITADEL = "https://data.statistiques.developpement-durable.gouv.fr/dido/api/v1/datafiles/8b35affb-55fc-4c1f-915b-7750f974446a/csv?DEP_CODE=in:{}&columns=COMM,ETAT_DAU,DATE_REELLE_AUTORISATION,SEC_CADASTRE1,NUM_CADASTRE1,NB_LGT_TOT_CREES"
CADASTRE = "https://cadastre.data.gouv.fr/data/etalab-cadastre/latest/geojson/communes/{}/{}/cadastre-{}-{}.json.gz"

log = pd.read_csv(get(SITADEL.format(",".join(villes.dep)), "sitadel_logements.csv"), sep=";", dtype=str)
log = log[log.COMM.isin(communes.com)].astype({"NB_LGT_TOT_CREES": int}).reset_index(drop=True)

def centroides(com, couche):
    fc = json.loads(gzip.decompress(get(CADASTRE.format(com[:2], com, com, couche), f"cadastre/{com}_{couche}.json.gz").read_bytes()))
    g = gpd.GeoDataFrame.from_features(fc["features"], crs=4326)
    c = g.to_crs(2154).centroid.to_crs(4326)
    return pd.DataFrame({"cle": g.id.str[:5] + g.id.str[8:], "lat": c.y, "lon": c.x, "niveau_geo": couche})

pc = pd.concat([centroides(com, couche) for com in sorted(set(log.COMM)) for couche in ["parcelles", "sections"]]).drop_duplicates("cle").set_index("cle")
sec = log.SEC_CADASTRE1.fillna("").str.strip().str.zfill(2)
num = log.NUM_CADASTRE1.fillna("").str.strip().str.zfill(4)
log[["lat", "lon", "niveau_geo"]] = None
for cle in [log.COMM + sec + num, log.COMM + sec]:
    log[["lat", "lon", "niveau_geo"]] = log[["lat", "lon", "niveau_geo"]].fillna(pc.reindex(cle).set_axis(log.index))
save(log.astype({"lat": float, "lon": float}), "bronze_sitadel")
log.niveau_geo.fillna("non localisé").value_counts(normalize=True).round(3)

bronze_sitadel 52363


niveau_geo
parcelles       0.592
sections        0.351
non localisé    0.056
Name: proportion, dtype: float64

In [13]:
lg = pd.read_csv(zipfile.ZipFile(get("https://www.insee.fr/fr/statistiques/fichier/8268838/base-ic-logement-2021_csv.zip", "logement2021.zip")).open("base-ic-logement-2021.CSV"), sep=";", dtype={"IRIS": str, "COM": str}, usecols=["IRIS", "COM", "P21_RP", "P21_RP_VOIT1P"])
save(lg[lg.COM.isin(communes.com)], "bronze_logement_2021")

geo = gpd.GeoDataFrame(iris, geometry=iris.geometry_json.apply(lambda g: shape(json.loads(g))), crs=4326)
car = pd.read_parquet(get("https://static.data.gouv.fr/resources/revenus-pauvrete-et-niveau-de-vie-donnees-carroyees-dispositif-fichier-localise-social-et-fiscal-filosofi/20260309-120901/carreaux-200m-met-3035-2021.parquet", "filosofi_carreaux_200m_2021.parquet"), columns=["idcar_200m", "ind", "men", "men_pauv", "ind_snv"])
car[["y", "x"]] = car.idcar_200m.str.extract(r"N(\d+)E(\d+)").astype(float) + 100
boites = geo.to_crs(3035).dissolve("ville").bounds
car = pd.concat([car[car.x.between(b.minx, b.maxx) & car.y.between(b.miny, b.maxy)] for b in boites.itertuples()]).drop_duplicates("idcar_200m")
save(car, "bronze_carreaux")

qpv = gpd.read_file(f"zip://{get('https://static.data.gouv.fr/resources/quartiers-prioritaires-de-la-politique-de-la-ville-qpv/20260115-204323/qpv-2024-geojson.zip', 'qpv2024.zip')}!GEOJSON/QP2024_France_Hexagonale_Outre_Mer_WGS84.geojson")
qpv = qpv[qpv.insee_com.isin(communes.com)]
save(pd.DataFrame({"code_qp": qpv.code_qp, "lib_qp": qpv.lib_qp, "insee_com": qpv.insee_com, "geometry_wkt": qpv.geometry.to_wkt()}), "bronze_qpv")

bronze_logement_2021 1421


bronze_carreaux 55534


bronze_qpv 71
